# Favoritaの販売系列分析

時系列を訓練・検証・テストに分割し、検証MSE最小のモデルで予測誤差と選択構造を評価する。

GSE-VARはBernoulli構造ゲートと期待L0正則化を用い、DisARMでロジット勾配を推定する。実装はリポジトリの共通ライブラリから読み込む。

In [ ]:
from pathlib import Path
import sys

PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(PROJECT))


## 実行設定

TRAIN=TrueでGSE-VAR、cMLP、線形VARを学習する。実験ごとの設定と検証MSEによる選択手順はREADMEに記載する。

In [ ]:
from experiments.favorita.run import configuration
from experiments.favorita.study import execute

TRAIN = False
DEVICE = "cpu"
CONFIG = configuration()
CONFIG["seeds"] = [42]  # 論文で評価した初期値
CONFIG


In [ ]:
result_directories = []
if TRAIN:
    for method in ("GSE-VAR", "Linear-SGL"):
        result_directories.append(execute(PROJECT, CONFIG, method, device=DEVICE if method == "GSE-VAR" else "cpu"))


## cMLP比較

同一の訓練・検証・テスト区間を用い、20候補から検証MSE最小のモデルを選択する。

In [ ]:
from experiments.favorita.cmlp import main as run_cmlp

if TRAIN:
    result_directories.append(run_cmlp(["--project", str(PROJECT), "--data", str(PROJECT / "data/favorita/cmlp_inputs.npz"), "--device", DEVICE]))


## 集計

各実行の保存CSVを集計する。標準偏差は反復間の標本標準偏差であり、単一反復の場合は欠測となる。保存済み実行を読む場合は、`result_directories`に実行ディレクトリを指定する。

In [ ]:
from experiments.analyze import summarize

for directory in result_directories:
    display(summarize(directory))
